# 01 · Raw data and exploratory analysis

What the NEBSL workbook contains, how complete it is, and what the donor corneas look like. Code: `data_prep/eda.py`.

> **Privacy:** this notebook shows only counts, metrics, charts and study codes (R0001, Q0001), never NEBSL records. **Before committing, clear all outputs** (VS Code: `...` → *Clear All Outputs* → save).

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `recommendation` importable whether this runs from notebooks/, recommendation/ or ml/
ML = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "recommendation" / "config.py").exists())
sys.path.insert(0, str(ML))

import pandas as pd
from IPython.display import Image, display
from recommendation.config import load_config

pd.set_option("display.width", 160)
cfg = load_config()
DATASET = cfg["paths"]["matching_dataset"]
OUTPUTS = Path(cfg["paths"]["outputs"])
print("Dataset:", Path(DATASET).name)

## 1. Run the EDA (summaries and charts only)

In [ ]:
from recommendation.data_prep import eda

EDA_DIR = OUTPUTS / "eda"
s = eda.run(DATASET, str(EDA_DIR), raw_path=cfg["paths"]["raw_workbook"])
pd.Series({"records": s["raw_rows"], "columns": s["raw_cols"],
           "columns > 50% missing": s["raw_missing_over_50"],
           "grade entries to verify": s["grade_entries_not_clean"],
           "grade A+ / A / B": " / ".join(str(v) for v in s["grade_counts"].values())}).to_frame("value")

## 2. Missing values in the raw workbook

Columns over about 70% missing are too sparse to use as features.

In [ ]:
pd.Series(s["raw_missing_top"]).rename("% missing").to_frame()

## 3. Donor cornea measures (cleaned)

In [ ]:
pd.DataFrame(s["measures"]).T.rename(columns={"50%": "median"})

In [ ]:
for f in ["07_grade_distribution.png", "03_endothelial_measures.png", "08_tissues_per_year.png"]:
    display(Image(filename=str(EDA_DIR / f)))

## 4. Date-order checks and values to review with NEBSL

In [ ]:
display(pd.Series(s["date_order_issues"]).rename("records").to_frame())
pd.Series(s["outliers_for_review"]).rename("corneas outside plausible range").to_frame()

**Findings to note:** grade is driven mainly by ECD; requests exist almost only for 2024–2025; a few ECD/CV values and dates need checking with NEBSL (flagged, not deleted).